In [ ]:
### firstly extracting decent question out of user query. . . .

In [18]:
from crewai.tools import tool
from groq import Groq

import os
import json
import requests
from bs4 import BeautifulSoup
import fitz
import requests
import tempfile


In [19]:
class ResearchStore:

    def __init__(self):

        self.wikipedia = {}
        self.serper = {}
        self.documents = []
        self.facts = []

    def add_wikipedia(self, query, results):

        self.wikipedia[query] = results

        self.documents.extend(results)

    def add_serper(self, query, results):

        self.serper[query] = results

    def add_documents(self, docs):

        self.documents.extend(docs)

    def add_facts(self,facts):
        self.facts.extend(facts)

    # Debug Helpers

    def summary(self):

        return {
            "wikipedia_queries":
                len(self.wikipedia),

            "serper_queries":
                len(self.serper),

            "documents":
                len(self.documents),

            "facts":
                len(self.facts)
        }

### generate queries

In [77]:
client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

def generate_research_queries(user_query: str) -> dict:
    """
    Converts a complex urban transport planning query
    into focused research/search queries.

    Args:
        user_query (str): User's transport planning request

    Returns:
        dict: Structured research queries
    """

    SYSTEM_PROMPT = """
        You are an urban transport research planner and information retrieval expert.

        Your task is to convert a user's urban transport planning request into highly effective search queries for data collection.

        The generated queries will be used with:
        - Wikipedia Search
        - Google Search (Serper)

        Rules:

        1. Generate search-engine friendly queries.
        2. Always include the city name when possible.
        Example:
        - "Lucknow population growth"
        - NOT "population growth"

        3. Prioritize queries that are likely to return:
        - numerical values
        - statistics
        - percentages
        - growth rates
        - ridership figures
        - budgets
        - project costs
        - transport indicators
        - official planning data
        - government reports

        4. Focus on information needed for transport planning:

        - population and demographics
        - population growth and future population projections
        - employment and economic activity
        - land use patterns and major activity centers
        - transport infrastructure
        - public transport performance and ridership
        - travel demand and commuting patterns
        - transport mode share (modal split)
        - vehicle ownership and motorization rates
        - traffic congestion and bottlenecks
        - sustainability and non-motorized transport
        - current and planned transport projects
        - transport policies and mobility initiatives

        5. Avoid vague or ambiguous queries.

        6. Queries should be directly usable in Google or Wikipedia search.

        7. Generate 2 high-quality queries per category.

        Return ONLY valid JSON.

        Output format:

        {
        "demographics": [],
        "economics": [],
        "land_use": [],
        "transport_infrastructure": [],
        "mobility_patterns": [],
        "traffic_congestion": [],
        "environment_sustainability": [],
        "policies_projects": [],
        "financials": []
        }
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_query
            }
        ]
    )

    queries = json.loads(
        response.choices[0].message.content
    )

    return queries

In [78]:
question_return = generate_research_queries(
    "Reduce traffic congestion in Bangalore."
)

print(json.dumps(question_return, indent=3))

{
   "demographics": [
      "Bangalore population growth rate",
      "Bangalore urban demographics"
   ],
   "economics": [
      "Bangalore GDP growth",
      "Bangalore economic activity centers"
   ],
   "land_use": [
      "Bangalore land use patterns",
      "Bangalore urban sprawl"
   ],
   "transport_infrastructure": [
      "Bangalore transport infrastructure development",
      "Bangalore road network expansion"
   ],
   "mobility_patterns": [
      "Bangalore commuting patterns",
      "Bangalore transport mode share"
   ],
   "traffic_congestion": [
      "Bangalore traffic congestion hotspots",
      "Bangalore traffic volume statistics"
   ],
   "environment_sustainability": [
      "Bangalore non-motorized transport initiatives",
      "Bangalore sustainable urban mobility plans"
   ],
   "policies_projects": [
      "Bangalore traffic management policies",
      "Bangalore smart traffic management projects"
   ],
   "financials": [
      "Bangalore transport infrastruc

In [ ]:
all_questions=[]

for key, value in question_return.items():
    for i in value:
        all_questions.append(i)
    

### LLM Checking

In [128]:
def check(user_query : str)-> str:

    PROMPT=f''' 
    You are a senior traffic analyst. Tell me what kind of data are you seeing and study this {user_query} 
    and tell me your best insights about this. 
    Straight up insights no unnecessary heading before giving insights.

    
    '''

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        # response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": PROMPT
            },
            {
                "role": "user",
                "content": user_query
            }
        ]
    )

    ans = response.choices[0].message.content
    

    return ans

In [129]:
q="Bengaluru Traffic Police - Congestion-map login Kannada visually challenged a Bengaluru Traffic Police Karnataka State Police, Govt. of Karnataka × About US Coffee Table Book Vision Mission History Commissioner's Message Joint Commissioner's Message Administrative Structure Rewards and recognition Traffic Management Goal & Objectives Team OFFICE OF TRAFFIC PLANNING Rules & Regulation Systems Traffic Management Centre ASTraM Adaptive Traffic Control System (ATCS) Social Media Cell INITIATIVES Ambulance Priority Junction Improvement Implementation & Results Components ITMS CAMERA'S Traffic Signal Light Camera PDA Cobra & MDT Safety Gadgets Street Furniture Variable Message Signs Enforcement GOAL & OBJECTIVE Rules & Regulations Spot Fines Team Field Enforcement Team Smart Enforcement Team Systems SMART ENFORCEMENT CENTER ITEMS E-CHALLAN SYSTEM INITIATIVES Special Drives Contactless Enforcement Contact Enforcement Statistics Black Spots Incident Management Road Safety Goal & Objectives Team TRAFFIC TRAINING & ROAD SAFETY INSTITUTE Student Association for Road Safety Traffic Warden Organisation Rules & Regulation Traffic Signs Advice to Drivers Systems i-RAD e-Path INITIATIVES Traffic Park Accident Statistics Road Safety ReportS Navigate Bengaluru RTI RTI Manual Rti Dashboard RTI Officers Contact Us FAQ Pay Traffic Fine Feedback Back Congestion Map The congestion map offers a comprehensive overview of traffic congestion across Bengaluru City. Congestion levels are assessed based on queue length, defined as the cumulative distance that congestion extends from a specific location on the road, such as from an intersection to the point where the traffic flow normalizes. Congestion levels are categorized as follows: • Severe: Queue length exceeds 750 meters. • High: Queue length ranges from 500 to 750 meters. • Moderate: Queue length ranges from 250 to 500 meters. Screen Reader Access The website complies with World Wide Web Consortium (W3C) Web Content Accessibility Guidelines (WCAG) 2.0 level AA. This will enable people with visual impairments access the website using assistive technologies, such as screen readers. The information of the website is accessible with different screen readers, such as JAWS, NVDA, SAFA, Supernova and Window-Eyes. Following table lists the information about different screen readers: Sl.No Screen Reader Website Free/ Commercial 1 Non Visual Desktop Access (NVDA) http://www.nvda-project.org/ (External website that opens in a new window) Free 2 JAWS http://www.freedomscientific.com (External website that opens in a new window) Commercial 3 Window-Eyes http://www.gwmicro.com (External website that opens in a new window) Commercial 4 System Access To Go http://www.satogo.com/ (External website that opens in a new window) Free 5 WebAnywhere http://webinsight.cs.washington.edu/ (External website that opens in a new window) Free Close Disclaimer Please note that this page also provides links to the websites / web pages of Govt. Ministries/Departments/Organisations. The content of these websites are owned by the respective organisations and they may be contacted for any further information or suggestion Website Policies and Guidelines Copyright Policy Hyperlinking Policy Security Policy Terms & Conditions Privacy Policy Accessibility Resources Sitemap Help Screen Reader Access Guidelines Last Updated: 2026-05-16 14:42:46 Visitors Counter: 1044726 Version: CeG/KRN 4.0 Content Owned and maintained by: Bengaluru Traffic Police , Karnataka State Police, Govt. of Karnataka Designed , Developed & Hosted by: Centre for e-governance - Webportal karnataka government © 2026, All Rights Reserved. Copyright Policy Close Copyright policy (if the published information is available free of any charge) 1)Information featured on this website can be re-published in any media form free of any charge only with prior permission from us(CeG) or concerned respective authority which is owning the website, through email. 2)Information can be republished as it is available and not to be used in a distorted or misleading manner. 3)Where the material is being published or suggested to others, the source must be prominently acknowledged 4)13. However, the permission to reproduce this material does not extend to any material on this site, which is explicitly identified as being the copyright of a third party Copyright policy (if there is a provision to reuse published information) 1)The information published on the website comes under copyright policy and obtaining authorization for their republish is a pre-requisite. 2)To get permission one can mail to ………………..@......... Hyperlinking Policy Close (in case the respective department doesn’t need any permission to hyperlink in their website) 1)We do not object to you linking directly to the information that is hosted on our site and no prior permission is required for the same. 2)we do not permit our pages to be loaded into frames on other sites. Our Department’s pages must loa"
print(check(q))

The Bengaluru Traffic Police website provides a comprehensive overview of traffic congestion across the city, with a congestion map that categorizes congestion levels as severe, high, or moderate based on queue length. The website is accessible to people with visual impairments, with features such as screen reader access and compatibility with various screen readers like JAWS, NVDA, and Window-Eyes. The website also provides information on traffic management, road safety, and enforcement, including details on ambulance priority, junction improvement, and e-challan systems. The data suggests that the traffic police are using technology to manage traffic and improve road safety, with initiatives such as adaptive traffic control systems, intelligent transportation management systems, and smart enforcement centers. The website also provides statistics on traffic accidents and road safety, which can be useful for analyzing trends and identifying areas for improvement. Overall, the website p

### now moving all these question to wikipedia OR serper. . .
### router function

In [97]:
# WIKI_KEYWORDS = [
#     # demographics
#     "population",
#     "demographics",
#     "age distribution",
#     "population growth",
#     "density",

#     # city profile
#     "history",
#     "geography",
#     "city overview",
#     "urban growth",

#     # economics
#     "economy",
#     "major industries",
#     "employment",

#     # infrastructure
#     "infrastructure",
#     "transport infrastructure",
#     "metro",
#     "bus system",
#     "railway",

#     # land use
#     "land use",
#     "commercial areas",
#     "residential areas"
# ]


# SERPER_KEYWORDS = [
#     # traffic
#     "traffic",
#     "congestion",
#     "bottleneck",
#     "road safety",

#     # transport demand
#     "ridership",
#     "mode share",
#     "modal split",
#     "travel demand",
#     "commuting pattern",
#     "household travel survey",

#     # vehicles
#     "vehicle ownership",
#     "registered vehicles",

#     # sustainability
#     "pollution",
#     "air quality",
#     "mobility",
#     "cycling",
#     "walkability",

#     # planning
#     "policy",
#     "project",
#     "government scheme",
#     "master plan",
#     "development plan",
#     "smart city",

#     # future
#     "future plan",
#     "future projection",
#     "population projection",

#     # economics
#     "budget",
#     "investment",
#     "construction cost",
#     "transport funding",

#     # infrastructure projects
#     "construction",
#     "expansion",
#     "corridor",
#     "metro extension",
#     "brt"
# ]


# # @tool("Route Research Queries")
# def route_research_queries(query_groups):

#     routed = {
#         "wikipedia": [],
#         "serper": []
#     }

#     for category, queries in query_groups.items():

#         for query in queries:

#             q = query.lower()

#             wiki_score = sum(
#                 keyword in q
#                 for keyword in WIKI_KEYWORDS
#             )

#             serper_score = sum(
#                 keyword in q
#                 for keyword in SERPER_KEYWORDS
#             )

#             if serper_score > wiki_score:
#                 routed["serper"].append(query)

#             else:
#                 routed["wikipedia"].append(query)

#     return routed



In [98]:
# routed=route_research_queries(question_return)
# routed['wikipedia']

### extract city name

In [86]:
def extract_city_name(user_query : str)-> str:

    CITY_PROMPT=f''' 
    from inside the {user_query} fetch the city name. Just the city name NOTHING ELSE.
    '''

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        # response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": CITY_PROMPT
            },
            {
                "role": "user",
                "content": user_query
            }
        ]
    )

    city_name = response.choices[0].message.content
    

    return city_name
    

In [87]:
extract_city_name("Banglore employement rate")

'Bangalore'

### wiki search

In [81]:
WIKIPEDIA_HEADERS = {
    "User-Agent": "UrbanTransportAgent/1.0 (aryasen1212@gmail.com)"
}

WIKIPEDIA_BASE_URL = "https://en.wikipedia.org/w/api.php"


# @tool("Wikipedia Search Tool")
def search_wikipedia(query: str, limit: int = 3) -> list:
    """
    Searches Wikipedia and returns top relevant pages
    with extracts/snippets.

    Args:
        query (str):
            Search query

        limit (int):
            Number of results

    Returns:
        list:
        [
            {
                "title": "",
                "snippet": "",
                "pageid": "",
                "source": "wikipedia"
            }
        ]
    """

    # =========================
    # STEP 1: SEARCH WIKIPEDIA
    # =========================

    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": limit
    }

    response = requests.get(
        WIKIPEDIA_BASE_URL,
        headers=WIKIPEDIA_HEADERS,
        params=search_params
    )

    data = response.json()
    # print("Wikipedia Search Response:")
    # print(json.dumps(data, indent=2))
    # print("="*60)
    # print("\n\n\n")

    search_results = data["query"]["search"]

    final_results = []

    # =========================
    # STEP 2: FETCH EXTRACTS
    # =========================

    for result in search_results:

        title = result["title"]

        extract_params = {
            "action": "query",
            "prop": "extracts",
            "titles": title,
            "explaintext": 1,
            "format": "json"
        }

        extract_response = requests.get(
            WIKIPEDIA_BASE_URL,
            headers=WIKIPEDIA_HEADERS,
            params=extract_params
        )

        extract_data = extract_response.json()

        pages = extract_data["query"]["pages"]

        # print("Pages:")
        # print(pages)
        # print("---"*60)
        # print("\n\n\n")

        page_data = list(pages.values())[0]

        extract = page_data.get("extract", "")

        final_results.append({
            "title": title,
            "content": extract[:5000],  # first 5000 chars
            # "pageid": result["pageid"],
            "source": "wikipedia"
        })

    return final_results

In [100]:
user_query="create a sustainable plan for lucknow"
city_name=extract_city_name(user_query)
print(city_name,"\n")
i=city_name+" city"
print(i)
print(json.dumps(search_wikipedia(i, limit=3), indent=2))


Lucknow 

Lucknow city
[
  {
    "title": "Lucknow",
    "content": "Lucknow (Hindi: Lakhana\u016b, pronounced [\u02c8l\u0259k\u02b0n\u0259.u\u02d0] ) is a metropolis and the second largest city of the Indian state of Uttar Pradesh where it serves as the capital and the administrative headquarters of the eponymous district and division. The city had a population of 2.8 million according to the 2011 census making it the eleventh most populous city and the twelfth-most populous urban agglomeration of India. It is an important centre of education, commerce, aerospace, finance, pharmaceuticals, information technology, design, culture, tourism, music, and poetry. Lucknow, along with Agra and Varanasi, forms the backbone of the Uttar Pradesh Heritage Arc.\nIn the 6th century BCE, Lucknow was part of Kosala, one of the 16 Mahajanapadas during the late Vedic period. The Nawabs of Lucknow acquired the name after the reign of the third Nawab, when Lucknow became their capital. In 1856, the East 

In [41]:

# query="Delhi population growth rate"
# wiki_results = search_wikipedia(query)

# store.wikipedia[query] = wiki_results

In [ ]:
for key, value in question_return.items():
    for i in value:
        print(i)

Bangalore population growth rate
Bangalore urban demographics
Bangalore GDP growth
Bangalore economic activity centers
Bangalore land use patterns
Bangalore urban sprawl
Bangalore transport infrastructure development
Bangalore road network expansion
Bangalore commuting patterns
Bangalore transport mode share
Bangalore traffic congestion hotspots
Bangalore traffic volume statistics
Bangalore non-motorized transport initiatives
Bangalore sustainable urban mobility plans
Bangalore traffic management policies
Bangalore smart traffic management projects
Bangalore transport infrastructure budget
Bangalore traffic congestion reduction costs


### serper API search

In [106]:
all_questions

['Bangalore population growth rate',
 'Bangalore urban demographics',
 'Bangalore GDP growth',
 'Bangalore economic activity centers',
 'Bangalore land use patterns',
 'Bangalore urban sprawl',
 'Bangalore transport infrastructure development',
 'Bangalore road network expansion',
 'Bangalore commuting patterns',
 'Bangalore transport mode share',
 'Bangalore traffic congestion hotspots',
 'Bangalore traffic volume statistics',
 'Bangalore non-motorized transport initiatives',
 'Bangalore sustainable urban mobility plans',
 'Bangalore traffic management policies',
 'Bangalore smart traffic management projects',
 'Bangalore transport infrastructure budget',
 'Bangalore traffic congestion reduction costs']

In [131]:
SERPER_API_KEY = os.environ["serper_api_key"]

def extract_pdf_text(pdf_url):

    response = requests.get(pdf_url)

    with tempfile.NamedTemporaryFile(
        suffix=".pdf",
        delete=False
    ) as temp_pdf:
        temp_pdf.write(response.content)
        temp_pdf_path = temp_pdf.name

    doc = fitz.open(temp_pdf_path)

    full_text = ""

    for page in doc:
        full_text += page.get_text()
    doc.close()

    # delete the temp file
    os.remove(temp_pdf_path)
    return full_text[:5000]



# @tool("Serper Search Tool")
def search_serper(query: str, limit: int = 3) -> list:
    """
    Searches Google using Serper API.

    Useful for:
    - current transport issues
    - policies
    - traffic congestion
    - news
    - infrastructure projects

    Args:
        query (str):
            Search query

        limit (int):
            Number of results

    Returns:
        list:
        [
            {
                "title": "",
                "snippet": "",
                "link": "",
                "source": "serper"
            }
        ]
    """

    url = "https://google.serper.dev/search"

    payload = {
        "q": query,
        "num": limit,
        "gl": "in"
    }

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        json=payload,
        headers=headers
    )

    data = response.json()

    organic_results = data.get("organic", [])

    final_results = []

    for result in organic_results:

        final_results.append({
            "title": result.get("title", ""),
            "content": result.get("snippet", ""),
            "link": result.get("link", ""),
            "source": "serper"
        })

    return final_results


# @tool("Fetch Serper Content")
def fetch_serper_content(
    serper_results: list,  # because all the serper result comes in a list of dict.
    top_k: int = 3
) -> list:
    """
    Fetches webpage content from top ranked
    Serper search results.

    Args:
        serper_results (list):
            Output from search_serper()

        top_k (int):
            Number of top results to fetch

    Returns:
        list:
        [
            {
                "title": "",
                "snippet": "",
                "link": "",
                "source": "serper"
            }
        ]
    """


    # take top 3 results from the list of serpi_all
    top_results = serper_results[:top_k]    # not considering some good domains to fetch the result. . . 

    # FETCH CONTENT
    # =========================

    final_results = []

    for item in top_results:
        result = item
        try:

            # =========================
            # PDF CASE
            # =========================

            if result["link"].lower().endswith(".pdf"):
                text=extract_pdf_text(result["link"])
            
            # =========================
            # HTML CASE
            # =========================

            else:
                response = requests.get(
                    result["link"],
                    timeout=10,
                    headers={
                        "User-Agent":
                        "Mozilla/5.0"
                    }
                )
                soup = BeautifulSoup(
                    response.text,
                    "lxml"
                )
                # remove unwanted tags
                for tag in soup([
                    "script",
                    "style",
                    "nav",
                    "footer",
                    "header"
                ]):
                    tag.decompose()

                text = soup.get_text(
                    separator=" ",
                    strip=True
                )

                # clean excessive whitespace
                text = " ".join(text.split())
            
            
            final_results.append({
                "title": result["title"],
                "link": result["link"],
                "content": text[:5000],  # limit chars ---------------------------------CHECK 
                "source": "serper"
            })

        except Exception as e:

            final_results.append({
                "title": result["title"],
                "link": result["link"],
                "content": f"ERROR: {str(e)}",
                "source": "serper"
            })

    return final_results

In [122]:
# haha=search_serper("Bangalore traffic volume statistics")
# print(haha)

In [132]:
doc=fetch_serper_content(search_serper("Bangalore road network expansion"))

In [133]:
doc

[{'title': '[PDF] Road Networks of Bengaluru (Urban) - KSRSAC',
  'link': 'https://kgis.ksrsac.in/bgis/pluggins/files/TransportationNetworks_5.pdf',
  'content': "Hosur Road\nMagadi Main Rd\nKanakapura Road\nBannerghata-Anekal Road\nOld Madras Road\nSH 104\nService Road\nNICE Ring Rd (Toll road)\nMysore Road\nDodda Aladmara Road\nHesarghatta Main Road\nSH-35\nNICE RING ROAD TOLL ROAD\nDoddaballapur Road\nOuter Ring Road\nPipeline Road\nITPL Main Road\nAttibele - Anekal Road\nIVRI Road\nPipe Line Road\nNH 4 to Budigere\nSarjapur Rd\nSarjapura - Attibele Rd\nNH7 Service Road\nBangalore - Hyderabadh Road\nNICE Ring Rd(Toll Road)\nChord Road\nHuskur Road\nMagadi Main Road\nBengaluru-Pune National Highway\nNelamangala - Majestic Service Rd\nHesaraghatta Road\nChandapura - Anekal Road\nNelamangala Road\nHebbal Ring Road\nAnekal - Hosur Road\nIttanguru Rd\nAgara Main Road\nHosa Road\nMuthanallur Rd\nHarohalli Road\n100 Feet Road\nBegur Road\nGatthalli Road\nHAL Old Airport Road\nWhitefield Ma

[{'title': 'Bengaluru Traffic Police - Index', 'content': 'Pay Traffic Challans · Report Violation · Raise Complaint · Be a Traffic Warden · Be Aware Be Smart.', 'link': 'https://btp.karnataka.gov.in/en', 'source': 'serper'}, {'title': 'Barco video wall transforms Bengaluru traffic management', 'content': 'To address these challenges, the Bengaluru Traffic Police (BTP) initiated the B-TRAC modernization plan, which included setting up a Traffic ...', 'link': 'https://www.barco.com/en/inspiration/customer-stories/control-room/bengaluru-traffic-police-india', 'source': 'serper'}, {'title': '[PDF] “B-TRAC – Technology Driven Traffic Management”', 'content': 'Some of the measures taken by Bangalore Traffic Police are given below,. Page 24. 22. 1) Effective Enforcement of Traffic Rules: Bangalore ...', 'link': 'https://bprd.nic.in/uploads/pdf/201702101040338492118BTRACcoverpage.pdf', 'source': 'serper'}]


In [ ]:
def extract_key_facts(documents):
    SYSTEM_PROMPT = """
    You are a senior traffic analyst. Tell me what kind of data are you seeing and study it.
    and tell me your best insights about this.  Straight up insights no unnecessary heading before giving insights.
    
    ALSO,

    Your task is to extract important factual information
    from urban transport and demographic documents.

    Rules:

    Extract transport planning metrics.

    Prioritize following items only when they are present in the content DO NOT made data by YOURSELF:

    - population
    - population growth rate
    - density
    - employment
    - vehicle ownership
    - mode share
    - ridership
    - trip rates
    - congestion indicators
    - major corridors
    - planned projects

    Return structured JSON and ONLY valid JSON.

    Format:
        {
            "insights": " ",
            "key_facts": [
                "...",
                "...",
                "..."
            ]
        }

    """
    
    extracted_facts = []

    for doc in documents:

        title = doc["title"]
        content = doc["content"]

        

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            response_format={
                "type": "json_object"
            },
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": content
                }
            ]
        )

        facts = json.loads(
            response.choices[0].message.content
        )

        extracted_facts.append({

            "title": title,

            "source": doc["source"],
            "insights": facts.get("insights",[]),

            "key_facts": facts.get("key_facts", [])
        })

    return extracted_facts

In [148]:
extract_key_facts(doc)

[{'title': '[PDF] Road Networks of Bengaluru (Urban) - KSRSAC',
  'source': 'serper',
  'insights': 'The data provided appears to be a comprehensive list of road names in and around Bangalore, India, suggesting a focus on urban transport planning and infrastructure development. The sheer number of roads listed indicates a complex network that requires careful management and planning to ensure efficient traffic flow and minimize congestion.',
  'key_facts': ['Over 300 road names are listed, indicating a large and complex road network',
   'The presence of multiple highways (e.g., Bengaluru-Pune National Highway, NH 4, NH 7) and toll roads (e.g., NICE Ring Rd) suggests significant investment in high-capacity roads',
   'Several major corridors are identifiable, including Hosur Road, Kanakapura Road, and Outer Ring Road, which are likely to be key routes for commuters and freight',
   'The diversity of road names suggests a mix of urban, suburban, and rural areas, implying varied transpor

### AGENT #1  orchestration to call in single place

In [ ]:
def run_data_fetching_agent(
    user_query
):

    store = ResearchStore()

    queries = generate_research_queries(
        user_query
    )

    # routed = route_research_queries(
    #     queries
    # )

    # for query in routed["wikipedia"]:
    city_name=extract_city_name(user_query)

    wiki_results = search_wikipedia(city_name + " city")

    store.add_wikipedia(
        query,
        wiki_results
    )
    
    for query in all_questions:

        serper_results = search_serper(
            query
        )

        store.add_serper(
            query,
            serper_results
        )

        docs = fetch_serper_content(
            serper_results
        )

        store.add_documents(
            docs
        )

    facts = extract_key_facts(
        store.documents
    )

    store.add_facts(
        facts
    )

    return store

In [48]:
store = run_data_fetching_agent(
    "Reduce traffic congestion in Bangalore."
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k8na9y9repb8ywr8g5nhx23w` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98740, Requested 1342. Please try again in 1m10.848s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [19]:
print(
    store.summary()
)

{'wikipedia_queries': 5, 'serper_queries': 10, 'documents': 45, 'facts': 45}


In [20]:
print(
    json.dumps(store.wikipedia,indent=2)
    )

{
  "Delhi India population density": [
    {
      "title": "List of states and union territories of India by population",
      "content": "India is a union consisting of 28 states and 8 union territories. As of 2026, with an estimated population of 1.47 billion, India is the world's most populous country. India occupies 2.4% of the world's area and is home to 17.5% of the world's population. The Indo-Gangetic Plain has one of the world's biggest stretches of fertile not-deep alluvium and are among the most densely populated areas of the world. The eastern and western coastal regions of Deccan Plateau are also densely populated regions of India. The Thar Desert in western Rajasthan is one of the most densely populated deserts in the world. The northern and north-eastern states along the Himalayas contain cold arid deserts with fertile valleys. These states have relatively low population density due to indomitable physical barriers.\n\n\n== Census of India ==\n\nThe first population c

In [21]:
store.facts

[{'title': 'List of states and union territories of India by population',
  'source': 'wikipedia',
  'key_facts': ['India has 28 states and 8 union territories',
   "India's estimated population is 1.47 billion as of 2026",
   "India occupies 2.4% of the world's area",
   "17.5% of the world's population lives in India",
   'The first population census in British India was conducted in 1872',
   'A census has been conducted every 10 years in India since 1947',
   'India has 641,000 inhabited villages',
   '72.2% of the total population resides in rural areas',
   'Dadra and Nagar Haveli and Daman and Diu has the fastest growth rate of 55.1 percent',
   'Nagaland recorded the lowest growth rate of -0.5 percent']},
 {'title': 'Demographics of India',
  'source': 'wikipedia',
  'key_facts': ["India's population reached 1.428 billion in April 2023",
   'Median age in India is approximately 29.8 years as of 2024',
   '68% of the population is between 15 and 64 years old',
   'Population gro

In [ ]:
with open("banglore.json", "w") as f:
    json.dump(store.facts, f, indent=2)

In [23]:
''' Stored the agent 1 final output in facts.json '''

' Stored the agent 1 final output in facts.json '

## Agent 2 Starting

In [1]:
''' Make a class to store all the info and be able to call it afterwards. . . . '''

class AnalysisStore():

    def __init__(self):

        self.mobility_patterns = []

        self.current_demand = {}

        self.future_demand = {}

        self.capacity_gaps = []

        self.bottlenecks = []

        self.priority_corridors = []

        self.recommended_focus_areas = []

In [2]:
# call the agent 1 data
import json
with open("facts.json") as f:
    facts = json.load(f)

In [6]:
# print(type(facts))
facts[0]

{'title': 'List of states and union territories of India by population',
 'source': 'wikipedia',
 'key_facts': ['India has 28 states and 8 union territories',
  "India's estimated population is 1.47 billion as of 2026",
  "India occupies 2.4% of the world's area",
  "17.5% of the world's population lives in India",
  'The first population census in British India was conducted in 1872',
  'A census has been conducted every 10 years in India since 1947',
  'India has 641,000 inhabited villages',
  '72.2% of the total population resides in rural areas',
  'Dadra and Nagar Haveli and Daman and Diu has the fastest growth rate of 55.1 percent',
  'Nagaland recorded the lowest growth rate of -0.5 percent']}

In [10]:
# llm calling
from groq import Groq
client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

In [11]:
# import pprint
response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        # response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "helpful assistant"
            },
            {
                "role": "user",
                "content": "who are u?"
            }
        ]
    )

# pprint.pprint(response1)
print(response.choices[0].message.content)

I'm an artificial intelligence (AI) designed to assist and communicate with users like you. I'm a computer program that uses natural language processing (NLP) to understand and respond to questions, provide information, and engage in conversation. I don't have a personal identity or emotions, but I'm here to help with any questions or topics you'd like to discuss.

I can provide information on a wide range of subjects, from science and history to entertainment and culture. I can also help with tasks like language translation, math problems, and generating text. My goal is to be a helpful and informative resource for you, so please don't hesitate to ask me anything!

What would you like to talk about or ask? I'm all ears (or rather, all text)!


In [49]:
print(type(facts))

<class 'list'>


In [12]:
def run_analysis_agent(facts: list)-> dict:
    SYSTEM_PROMPT_ANALYST = """
You are a SENIOR TRANSPORT DEMAND ANALYST with 15+ years experience in urban mobility.

Your task: Transform raw city data into ACTIONABLE transport insights.

INPUT: City profile with population, area, existing infrastructure, known problems
OUTPUT: Detailed demand analysis that a transport designer can use

ANALYSIS FRAMEWORK:

1. MOBILITY PATTERNS (200-250 words)
   - How do people currently move around the city?
   - Commute patterns (home → work, home → school)
   - Peak hour characteristics
   - Mode share (% using cars, buses, walking, motorcycles)
   - Trip distribution by time of day
   - Key origin-destination pairs (where do most people go?)
   
2. CURRENT TRANSPORT DEMAND (150-200 words)
   - Total daily trips (calculate: population × avg trips per person per day)
   - Current public transport share (%)
   - Current private vehicle share (%)
   - Unmet demand (people who want to use PT but can't)
   - Daily commuters vs occasional travelers
   - Peak hour volume estimates
   
3. FUTURE DEMAND PROJECTION (150-200 words)
   - Project demand for next 5 and 10 years
   - Account for population growth rate
   - Account for urbanization/motorization trends
   - Expected mode shift (if PT improves, % using it increases)
   - Estimated future daily trips
   - Future peak hour demands
   
4. CAPACITY GAPS (200-250 words)
   - Current PT capacity vs demand gap
   - How many people can existing buses handle daily?
   - How many trips are unserved?
   - Which time periods are most critical?
   - Geographic gaps (areas with no PT coverage)
   - Income-based gaps (afford cars vs need PT)
   - Example: "Current buses serve 400k/day, but 800k+ need PT → Gap of 400k"
   
5. BOTTLENECKS & CONGESTION RISKS (200-250 words)
   - Specific locations where traffic converges
   - Why these are bottlenecks (geography, limited alternatives)
   - Congestion severity (current vs projected)
   - Impact on commute times
   - Economic losses (congestion cost per year if calculable)
   - Safety risks in congested areas
   - Example: "Gomti Nagar intersection: 500k vehicles/day trying to merge, only 4 lanes available"
   
6. PRIORITY TRANSPORT CORRIDORS (200-250 words)
   - Rank top 5 corridors by demand/need
   - For each corridor: origin → destination, distance, estimated daily demand
   - Current mode of transport on corridor
   - Why this corridor matters (connects major employment, residential, educational areas)
   - Current travel time vs acceptable travel time
   - Corridor characteristics (flat/hilly, dense/sparse)
   - Feasibility for mass transit (straight line? good ROI?)
   - Example: "Corridor 1: Residential Zone (500k people) ↔ IT Park (100k jobs), 15 km, 250k daily demand, currently 45 min by bus"
   
7. PUBLIC TRANSPORT DEFICIENCIES (200-250 words)
   - What's missing from current PT system?
   - Coverage gaps (areas without any buses)
   - Frequency gaps (buses come every 30 min, should be every 10 min)
   - Reliability issues (late, unreliable)
   - Comfort/safety issues
   - Last-mile connectivity (how to get from home to bus stop?)
   - Integration issues (buses don't connect with metro)
   - Cost barriers (too expensive for poor people)
   - Example: "Eastern zone has 200k population but only 3 bus routes. No metro access. Last bus at 9 PM."
   
8. DEMAND ELASTICITY & MODE SHIFT POTENTIAL (150-200 words)
   - If you improve PT, how many people will switch from cars?
   - If metro is added, what % increase in PT usage?
   - Price sensitivity (if bus cost drops by 20%, how many more riders?)
   - Quality sensitivity (if frequency improves, how many switch?)
   - Estimate potential mode shift if new infrastructure added
   - Example: "Metro in corridor would attract 30% of current car users + 40% of motorcycle users"

CRITICAL RULES:
- USE ONLY PROVIDED FACTS. Do not invent data.
- If data missing, STATE IT CLEARLY ("Population growth rate not provided")
- Base calculations on standard urban mobility benchmarks where appropriate
- Show your reasoning (e.g., "Assuming average 2.5 trips/person/day...")
- Be specific with numbers, not vague
- Include confidence levels ("high confidence", "estimated", "rough projection")
- Format as structured JSON with each section as detailed text, NOT bullet points

CALCULATION EXAMPLES:
- Daily trips = Population × 2.5 (standard urban average)
- PT demand = Daily trips × current PT share
- Peak hour = Daily trips ÷ 12 hours × 2 (peak hours have 2x average)
MINIMUM DETAIL:
- Each section must be 150-250 words
- Include specific numbers and corridors
- Explain WHY (cause + effect reasoning)
- Give examples where possible

Return ONLY valid JSON. No explanations outside JSON.
"""
    
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT_ANALYST
            },
            {
                "role": "user",
                # "content": facts -> cant use direct list. . 
                "content": json.dumps(facts,indent=2)
            }
        ]
    )

    analysis=json.loads(response.choices[0].message.content)
    # analysis=json.loads(response)
    
    return analysis
    
    

In [13]:
a=json.dumps(run_analysis_agent(facts), indent=2)
# print(run_analysis_agent(facts))
print(a)

{
  "mobility_patterns": "People in Delhi currently move around the city using a combination of modes, including buses, metro, and private vehicles. The commute patterns are primarily home to work and home to school. Peak hour characteristics show a high volume of traffic, with a mode share of 33% using buses, 30% walking, and less than 10% using private cars. The trip distribution by time of day is not explicitly stated, but it can be inferred that the peak hours are during the morning and evening commutes. Key origin-destination pairs include residential areas to commercial areas, such as Hauz Khas to Connaught Place. Assuming an average of 2.5 trips per person per day, the total daily trips in Delhi can be estimated to be around 27.5 million (2.5 trips/person/day * 11 million population).",
  "current_transport_demand": "The total daily trips in Delhi are estimated to be around 27.5 million. The current public transport share is around 60%, with 33% using buses and 27% using the met

### trip calculation formulas
input -> entire fatcs

In [ ]:
def trip_calculation (facts : list ) -> list:
    pass

### cost benefit ratios

In [16]:
def cost_calculation (facts : list ) -> list:
    pass

In [ ]:
print('f')